# Step 6 — Antigenicity and Solvent Accessibility Scoring

**Pipeline:** Myocarditis structural mimicry reassessment (spike vs. cardiac autoantigens)
**This step:** determines whether the structurally similar regions identified in the fragment
analysis (Step 4) are antibody-accessible or buried.

**Reads:** `alignment_results.json` (Step 4, including corrected fragment analysis), spike and
TPM1 chain PDB files from Step 4
**Produces:** `antigenicity_results.json` — per-residue solvent accessibility for the
structurally similar regions, determining whether they are immunologically relevant

### The specific question this step answers

Step 4 found extensive local structural similarity between postfusion spike S2 and TPM1/MYH6
at the 30-residue fragment level (TM up to 0.998, LDDT up to 0.99). Step 5 showed this
similarity is not statistically significant against random proteins of comparable size, because
it is driven by shared coiled-coil secondary structure class. This step asks the final question:
regardless of whether the similarity is cardiac-antigen-specific, do the structurally similar
residues sit on antibody-accessible surfaces?

If yes: anti-spike antibodies targeting the postfusion S2 coiled-coil could plausibly
cross-react with cardiac coiled-coil proteins, not by specific mimicry but by structural
class, which is a different and potentially more interesting finding.

If no: the structural similarity is real but immunologically inert (buried interfaces
cannot be antibody targets), and the null result is complete at every level.

### Tool choice

Solvent-accessible surface area (SASA) is computed locally using FreeSASA rather than via
the ElliPro web API. FreeSASA directly measures the physical property ElliPro estimates
(residue surface exposure), avoids web API dependency risk from a Colab session, and
produces numeric SASA values reportable directly in the manuscript. The biological question
answered is identical.

In [ ]:
import subprocess, sys, os, importlib, json

for pkg in ("freesasa", "requests", "gemmi", "numpy"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import freesasa
import numpy as np
print("freesasa:", freesasa.getVersion() if hasattr(freesasa, 'getVersion') else 'installed')

In [ ]:
MOUNT_DRIVE = True
BASE_DIR = "/content"

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/myocarditis_pipeline"

STEP4_DIR = os.path.join(BASE_DIR, "step4")
STEP6_DIR = os.path.join(BASE_DIR, "step6")
os.makedirs(STEP6_DIR, exist_ok=True)

with open(os.path.join(STEP4_DIR, "alignment_results.json")) as f:
    step4 = json.load(f)

# Locate chain PDB files
CHAINS_DIR = os.path.join(STEP4_DIR, "chains")
spike_postfusion_pdb = os.path.join(CHAINS_DIR, "spike_postfusion_6XRA_chainA.pdb")
tpm1_pdb = os.path.join(CHAINS_DIR, "TPM1_chain.pdb")
myh6_pdb = os.path.join(CHAINS_DIR, "MYH6_chain.pdb")

for label, path in [("Spike postfusion", spike_postfusion_pdb),
                     ("TPM1", tpm1_pdb), ("MYH6", myh6_pdb)]:
    exists = os.path.exists(path)
    print(f"  {label}: {'found' if exists else 'MISSING'} at {path}")

# Extract the top fragment hits from corrected fragment analysis
frag = step4.get("fragment_analysis", {})
patches = frag.get("significant_patches", [])
print(f"\nFragment patches available: {len(patches)}")

## Compute per-residue SASA for spike postfusion and cardiac antigens

FreeSASA computes the solvent-accessible surface area for each residue, giving a direct
physical measure of how exposed each residue is to solvent (and therefore to antibody binding).
A residue with high relative SASA (> 25% of its maximum possible SASA) is considered
surface-exposed; below that, it is considered buried.

In [ ]:
import freesasa
import gemmi

# Maximum SASA reference values per amino acid (Tien et al. 2013, empirical)
MAX_SASA = {
    "ALA": 129.0, "ARG": 274.0, "ASN": 195.0, "ASP": 193.0, "CYS": 167.0,
    "GLN": 225.0, "GLU": 223.0, "GLY": 104.0, "HIS": 224.0, "ILE": 197.0,
    "LEU": 201.0, "LYS": 236.0, "MET": 224.0, "PHE": 240.0, "PRO": 159.0,
    "SER": 155.0, "THR": 172.0, "TRP": 285.0, "TYR": 263.0, "VAL": 174.0,
}

EXPOSED_THRESHOLD = 0.25  # relative SASA > 25% = surface-exposed


def compute_per_residue_sasa(pdb_path):
    """Compute per-residue absolute and relative SASA using FreeSASA.
    Returns a list of dicts with residue info and SASA values."""
    structure = freesasa.Structure(pdb_path)
    result = freesasa.calc(structure)

    # Map atom-level SASA to residue-level
    residue_sasa = {}
    for i in range(structure.nAtoms()):
        resname = structure.residueName(i).strip()
        resnum = structure.residueNumber(i).strip()
        chain = structure.chainLabel(i).strip()
        key = (chain, resnum, resname)
        if key not in residue_sasa:
            residue_sasa[key] = {"chain": chain, "resnum": resnum, "resname": resname, "sasa": 0.0}
        residue_sasa[key]["sasa"] += result.atomArea(i)

    # Compute relative SASA
    residues = []
    for key, info in sorted(residue_sasa.items(), key=lambda x: int(x[0][1]) if x[0][1].lstrip("-").isdigit() else 0):
        max_sasa = MAX_SASA.get(info["resname"], 200.0)
        rel_sasa = info["sasa"] / max_sasa if max_sasa > 0 else 0
        info["relative_sasa"] = round(rel_sasa, 4)
        info["exposed"] = rel_sasa > EXPOSED_THRESHOLD
        residues.append(info)

    return residues


print("Computing SASA for spike postfusion chain...")
spike_sasa = compute_per_residue_sasa(spike_postfusion_pdb)
print(f"  {len(spike_sasa)} residues, {sum(1 for r in spike_sasa if r['exposed'])} exposed (>{EXPOSED_THRESHOLD*100}% relative SASA)")

print("\nComputing SASA for TPM1 chain...")
tpm1_sasa = compute_per_residue_sasa(tpm1_pdb)
print(f"  {len(tpm1_sasa)} residues, {sum(1 for r in tpm1_sasa if r['exposed'])} exposed")

print("\nComputing SASA for MYH6 chain...")
myh6_sasa = compute_per_residue_sasa(myh6_pdb)
print(f"  {len(myh6_sasa)} residues, {sum(1 for r in myh6_sasa if r['exposed'])} exposed")

## Map fragment hits to residue-level exposure

For each top fragment hit from Step 4, identify the specific residue range on both the spike
and antigen sides, and calculate what fraction of those residues are surface-exposed.
A structurally similar region that is predominantly surface-exposed on BOTH sides is a
plausible cross-reactive epitope; one that is buried on either side is not.

In [ ]:
def get_exposure_for_range(sasa_list, start_idx, end_idx):
    """Get exposure statistics for a residue index range."""
    # Fragment indices are 0-based from the chain extraction
    subset = sasa_list[start_idx:end_idx]
    if not subset:
        return None
    n_exposed = sum(1 for r in subset if r["exposed"])
    mean_rel_sasa = np.mean([r["relative_sasa"] for r in subset])
    return {
        "n_residues": len(subset),
        "n_exposed": n_exposed,
        "fraction_exposed": round(n_exposed / len(subset), 3),
        "mean_relative_sasa": round(float(mean_rel_sasa), 4),
    }


# Map the SASA lookup by protein
sasa_lookup = {
    "postfusion": spike_sasa,
    "closed": None,      # we have these but postfusion is the focus
    "one_rbd_up": None,
}
antigen_sasa_lookup = {
    "TPM1": tpm1_sasa,
    "MYH6": myh6_sasa,
}

# Analyze top fragment hits (focus on postfusion + TPM1/MYH6, the key pairs)
TOP_N_PATCHES = 20
focus_patches = [p for p in patches
                 if p.get("spike_state") == "postfusion"
                 and p.get("antigen") in ("TPM1", "MYH6")
                 and p["tm_score"] <= 1.0 and p["aln_length"] >= 15]
focus_patches.sort(key=lambda p: p["tm_score"], reverse=True)
focus_patches = focus_patches[:TOP_N_PATCHES]

print(f"Analyzing top {len(focus_patches)} postfusion fragment hits for surface exposure:\n")
print(f"{'TM':>6}  {'Antigen':<8}  {'Spike range':<15}  {'Spike exposed':>14}  {'Spike SASA':>11}  "
      f"{'Antigen range':<15}  {'Antigen exposed':>16}  {'Antigen SASA':>13}")
print("-" * 120)

exposure_results = []
for p in focus_patches:
    # Parse residue ranges from fragment labels (e.g. "spike_postfusion_0075_0105")
    q_parts = p["query_fragment"].split("_")
    q_start, q_end = int(q_parts[-2]), int(q_parts[-1])

    t_parts = p["target_fragment"].split("_")
    t_start, t_end = int(t_parts[-2]), int(t_parts[-1])

    antigen = p["antigen"]

    spike_exp = get_exposure_for_range(spike_sasa, q_start, q_end)
    antigen_exp = get_exposure_for_range(antigen_sasa_lookup.get(antigen, []), t_start, t_end)

    result = {
        **p,
        "spike_exposure": spike_exp,
        "antigen_exposure": antigen_exp,
        "both_sides_exposed": (
            spike_exp is not None and antigen_exp is not None
            and spike_exp["fraction_exposed"] > 0.5
            and antigen_exp["fraction_exposed"] > 0.5
        ),
    }
    exposure_results.append(result)

    if spike_exp and antigen_exp:
        print(f"{p['tm_score']:>6.3f}  {antigen:<8}  "
              f"{q_start:04d}-{q_end:04d}       "
              f"{spike_exp['fraction_exposed']:>13.1%}  "
              f"{spike_exp['mean_relative_sasa']:>11.3f}  "
              f"{t_start:04d}-{t_end:04d}       "
              f"{antigen_exp['fraction_exposed']:>15.1%}  "
              f"{antigen_exp['mean_relative_sasa']:>13.3f}")

# Summary
both_exposed = [r for r in exposure_results if r["both_sides_exposed"]]
print(f"\n--- Summary ---")
print(f"Total patches analyzed: {len(exposure_results)}")
print(f"Both sides >50% exposed: {len(both_exposed)}")
if both_exposed:
    print(f"\nPatches where BOTH spike and antigen surfaces are predominantly exposed:")
    for r in both_exposed:
        q_parts = r["query_fragment"].split("_")
        t_parts = r["target_fragment"].split("_")
        print(f"  TM={r['tm_score']:.3f}  spike {q_parts[-2]}-{q_parts[-1]} "
              f"({r['spike_exposure']['fraction_exposed']:.0%} exposed) vs "
              f"{r['antigen']} {t_parts[-2]}-{t_parts[-1]} "
              f"({r['antigen_exposure']['fraction_exposed']:.0%} exposed)")
else:
    print("\nNo patches are predominantly surface-exposed on BOTH sides.")
    print("The structural similarity is real but immunologically inert:")
    print("buried residues cannot serve as antibody epitopes.")

## Overall surface exposure profile of the structurally similar regions

Beyond individual patches, this cell shows the overall exposure profile of the spike
postfusion S2 regions that consistently match cardiac antigens (the HR1/HR2 coiled-coil
regions) to determine whether the coiled-coil structural class similarity lands on
exposed or buried surfaces.

In [ ]:
# Aggregate: which spike residue ranges are repeatedly hit across multiple TPM1 fragments?
from collections import Counter

spike_residue_hit_count = Counter()
for p in patches:
    if p.get("spike_state") == "postfusion" and p.get("antigen") == "TPM1":
        if p["tm_score"] <= 1.0 and p["aln_length"] >= 15:
            q_parts = p["query_fragment"].split("_")
            q_start, q_end = int(q_parts[-2]), int(q_parts[-1])
            for i in range(q_start, q_end):
                spike_residue_hit_count[i] += 1

if spike_residue_hit_count:
    # Find the spike residues most frequently involved in TPM1 matches
    hotspot_residues = sorted(spike_residue_hit_count.keys())
    print(f"Spike postfusion residues involved in TPM1 fragment matches: "
          f"{min(hotspot_residues)}-{max(hotspot_residues)} "
          f"({len(hotspot_residues)} unique residues)")

    # Exposure of the hotspot region
    hotspot_sasa = [spike_sasa[i] for i in hotspot_residues if i < len(spike_sasa)]
    if hotspot_sasa:
        n_exp = sum(1 for r in hotspot_sasa if r["exposed"])
        mean_sasa = np.mean([r["relative_sasa"] for r in hotspot_sasa])
        print(f"\nHotspot region exposure:")
        print(f"  {n_exp}/{len(hotspot_sasa)} residues exposed ({n_exp/len(hotspot_sasa):.1%})")
        print(f"  Mean relative SASA: {mean_sasa:.3f}")
        if n_exp / len(hotspot_sasa) > 0.5:
            print("  -> This region is PREDOMINANTLY SURFACE-EXPOSED")
            print("     Anti-spike antibodies targeting this coiled-coil region could")
            print("     plausibly access structurally similar surfaces on cardiac coiled-coil")
            print("     proteins (TPM1, MYH6 tail domain).")
        else:
            print("  -> This region is PREDOMINANTLY BURIED")
            print("     The coiled-coil similarity exists but is not antibody-accessible.")

    # Same analysis for TPM1 side
    tpm1_residue_hit_count = Counter()
    for p in patches:
        if p.get("spike_state") == "postfusion" and p.get("antigen") == "TPM1":
            if p["tm_score"] <= 1.0 and p["aln_length"] >= 15:
                t_parts = p["target_fragment"].split("_")
                t_start, t_end = int(t_parts[-2]), int(t_parts[-1])
                for i in range(t_start, t_end):
                    tpm1_residue_hit_count[i] += 1

    tpm1_hotspot = sorted(tpm1_residue_hit_count.keys())
    tpm1_hotspot_sasa = [tpm1_sasa[i] for i in tpm1_hotspot if i < len(tpm1_sasa)]
    if tpm1_hotspot_sasa:
        n_exp_t = sum(1 for r in tpm1_hotspot_sasa if r["exposed"])
        mean_sasa_t = np.mean([r["relative_sasa"] for r in tpm1_hotspot_sasa])
        print(f"\nTPM1 matched region exposure:")
        print(f"  {n_exp_t}/{len(tpm1_hotspot_sasa)} residues exposed ({n_exp_t/len(tpm1_hotspot_sasa):.1%})")
        print(f"  Mean relative SASA: {mean_sasa_t:.3f}")

else:
    print("No postfusion-TPM1 patches to analyze.")

## Write `antigenicity_results.json`

In [ ]:
from datetime import datetime, timezone

antigenicity_output = {
    "pipeline_step": 6,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "generated_in": "Google Colab",
    "method": "FreeSASA per-residue solvent-accessible surface area",
    "exposure_threshold": EXPOSED_THRESHOLD,
    "exposure_threshold_note": (
        f"Residues with relative SASA > {EXPOSED_THRESHOLD*100}% of their amino acid type's "
        f"maximum possible SASA (Tien et al. 2013 empirical reference values) are classified "
        f"as surface-exposed. This directly measures the physical property ElliPro estimates "
        f"via its protrusion index."
    ),
    "n_patches_analyzed": len(exposure_results),
    "n_both_sides_exposed": len(both_exposed),
    "patch_exposure_details": exposure_results[:30],
    "spike_postfusion_overall": {
        "total_residues": len(spike_sasa),
        "exposed_residues": sum(1 for r in spike_sasa if r["exposed"]),
    },
    "tpm1_overall": {
        "total_residues": len(tpm1_sasa),
        "exposed_residues": sum(1 for r in tpm1_sasa if r["exposed"]),
    },
}

output_path = os.path.join(STEP6_DIR, "antigenicity_results.json")
with open(output_path, "w") as f:
    json.dump(antigenicity_output, f, indent=2, default=str)

print("antigenicity_results.json written to:", output_path)

---

### Interpretation guide

**If the structurally similar regions are surface-exposed on both spike and TPM1:**
The finding is that postfusion spike S2's coiled-coil architecture presents an antibody-accessible
surface that structurally resembles surfaces on cardiac coiled-coil proteins. This is not specific
molecular mimicry (Step 5 showed the signal is not cardiac-antigen-specific), but it predicts
that anti-spike antibodies targeting the postfusion HR1/HR2 coiled-coil could cross-react with
human coiled-coil proteins broadly, cardiac antigens included. This is a testable, hypothesis-generating
finding that connects the structural observation to a plausible immunological mechanism.

**If the regions are buried on either side:**
The three-layer finding is complete: whole-domain similarity is not significant (Step 5), fragment-level
similarity is real but driven by shared secondary structure class (Step 4 fragment analysis), and
the structurally similar regions are not antibody-accessible (this step). The molecular mimicry
hypothesis finds no support at any level of analysis, which is a clean, publishable negative result
that strengthens the eBioMedicine finding with a conformationally complete method.

Either outcome completes the analysis. Steps 7-10 (conformational state comparison, novelty check,
pipeline packaging, and manuscript writing) follow from here.